In [ ]:
import random
import pandas as pd
import numpy as np
import deepchem as dc
#from pubchempy import get_cids, get_compounds

#import tensorflow as tf
#physical_devices = tf.config.list_physical_devices('GPU')
#tf.config.experimental.set_memory_growth(physical_devices[0], enable=True)

import matplotlib.pyplot as plt
#import seaborn as sns
#sns.set()

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler

In [ ]:
def display_df(df):
    display(HTML(df.to_html()))
    return None

def numtype(x):
    try:
        float(x)
        return True
    except:
        return False
    
def clean_num(s):
    s = str(s).strip().replace(">", "").replace("<", "")
    return float(s)

In [ ]:
RT = 0.593

## AB42

In [ ]:
df00 = pd.read_csv("data/raw/AB42.csv")
print("df00", df00.shape)
df00["smiles"] = df00["Ligand SMILES"].copy()
df00["target"] = df00["Target Name"].copy()
df00["binding_const"] = df00["IC50 (nM)"].copy()
df00["binding_const"] = df00["binding_const"].fillna(df00["KI (nM)"])
df00["binding_const"] = df00["binding_const"].apply(clean_num)
display_df(df00.head())

In [ ]:
df01 = pd.read_csv("data/raw/AB40_42.csv")
print("df01", df01.shape)
df01["smiles"] = df01["Ligand SMILES"].copy()
df01["target"] = "AB40_42"
df01["binding_const"] = df01["AB42 IC50 (nM)"].copy()
df01["binding_const"] = df01["binding_const"].fillna(df01["AB42 KI (nM)"])
df01["binding_const"] = df01["binding_const"].apply(clean_num)
display_df(df01.head())

In [ ]:
cnames = ["smiles", "target", "binding_const"]
df02 = pd.concat([df00[cnames], df01[cnames]], axis=0, ignore_index=False)
print("df02", df02.shape)
dfc = df02.groupby("smiles", as_index=False).agg(count=("binding_const", "count"))
df02 = df02.merge(dfc, how="left", on="smiles")
del dfc

df02["dG"] = RT * np.log(df02["binding_const"]*1e-09)

# train/test
np.random.seed(42)
ii = df02["count"] == 1
df_ = df02.loc[ii, ["smiles"]].copy().sample(150)
test_ligands = df_["smiles"].values.tolist()
print("test_ligands", len(test_ligands))
del df_

df02["train_test"] = "train"
ii = df02["smiles"].isin(test_ligands)
df02.loc[ii, "train_test"] = "test"

display_df(df02.head())
df02.to_csv("data/process/clean_train_test_800.csv", index=False)

In [ ]:
df02["dG"].plot(kind="kde")

In [ ]:
ii = df02["train_test"] == "train"
df02.loc[ii, "dG"].plot(kind="kde")
ii = df02["train_test"] == "test"
df02.loc[ii, "dG"].plot(kind="kde")

In [ ]:
ii = df02["train_test"] == "test"
df02.loc[ii, "dG"].plot(kind="kde")

In [ ]:
ii = df00["KI (nM)"].isnull()
df00.loc[~ii].head()